### CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

#### 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

##### Comment Out the cell below after first installation.

In [1]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create virtual environment
# !uv venv .venv --seed

# # Install dependencies
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

In [2]:
# python3-dev is required for bitsandbytes. You can install it with `sudo apt install python3-dev`.
# need gcc and g++ for bitsandbytes. You can install them with `sudo apt install build-essential gcc g++`.

##### Run the cell below every time to activate the installed environment. 

In [3]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

#### 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [4]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
# GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768 # 81,920 tokens best performance

# os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
# ^ use to force gpu selection. My pc only has 1 gpu so whatever

#### 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [5]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

# print("\n── MCQ sample ──")
# print(json.dumps(mcq_sample, indent=2))
# print("\n── Free-form sample ──")
# print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)


#### 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [6]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. "
    "Solve the problem carefully and briefly and put your final answer within \boxed{}."
    "If there are multiple answers, put them in a single \\boxed{} separated by commas."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Choose the single best answer from the options provided. "
    "Output only \\boxed{<letter>}."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# # Verify with samples
# for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
#     sys_p, usr_p = build_prompt(item["question"], item.get("options"))
#     print(f"── {label} user prompt (first 200 chars) ──")
#     print(usr_p[:200], "...\n")

#### 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [7]:
# use transformers maybe in huggingface

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=True,
    gpu_memory_utilization=0.88,
    max_model_len=30000, # 16384
    trust_remote_code=True,
    max_num_seqs=4, # 256
    max_num_batched_tokens=16384, # was 32768
)

sampling_params = SamplingParams(
    max_tokens=16384, # was 32768
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

INFO 04-07 19:57:08 [utils.py:233] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 30000, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.88, 'max_num_batched_tokens': 16384, 'max_num_seqs': 4, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'model': 'Qwen/Qwen3-4B-Thinking-2507'}
INFO 04-07 19:57:08 [model.py:549] Resolved architecture: Qwen3ForCausalLM
INFO 04-07 19:57:08 [model.py:1678] Using max model len 30000
(EngineCore pid=7672) INFO 04-07 19:57:13 [core.py:105] Initializing a V1 LLM engine (v0.19.0) with config: model='Qwen/Qwen3-4B-Thinking-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Thinking-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=30000, download_dir=None, load_format=bitsandbytes, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, 

(EngineCore pid=7672) <frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=7672) <frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:00<00:00,  2.07it/s]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:00<00:00,  2.03it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  2.96it/s]
(EngineCore pid=7672) 


(EngineCore pid=7672) INFO 04-07 19:57:19 [gpu_model_runner.py:4820] Model loading took 2.7 GiB memory and 2.360812 seconds
(EngineCore pid=7672) INFO 04-07 19:57:21 [backends.py:1051] Using cache directory: /home/lemoncastle/.cache/vllm/torch_compile_cache/faba89a1e6/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=7672) INFO 04-07 19:57:21 [backends.py:1111] Dynamo bytecode transform time: 1.74 s
(EngineCore pid=7672) INFO 04-07 19:57:21 [backends.py:285] Directly load the compiled graph(s) for compile range (1, 16384) from the cache, took 0.698 s
(EngineCore pid=7672) INFO 04-07 19:57:21 [decorators.py:303] Directly load AOT compilation from path /home/lemoncastle/.cache/vllm/torch_compile_cache/torch_aot_compile/ffcb11a63f9486b7124ad2c25d9637b422647f9d61e5677ee93bcf8c6bb18bdd/rank_0_0/model
(EngineCore pid=7672) INFO 04-07 19:57:21 [monitor.py:48] torch.compile took 2.66 s in total
(EngineCore pid=7672) INFO 04-07 19:57:22 [monitor.py:76] Initial profiling/warmup run took

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 4/4 [00:00<00:00, 14.38it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 3/3 [00:00<00:00, 17.10it/s]


(EngineCore pid=7672) INFO 04-07 19:57:26 [gpu_model_runner.py:6046] Graph capturing finished in 1 secs, took 0.11 GiB
(EngineCore pid=7672) INFO 04-07 19:57:26 [gpu_worker.py:597] CUDA graph pool memory: 0.11 GiB (actual), 0.14 GiB (estimated), difference: 0.03 GiB (25.9%).
(EngineCore pid=7672) INFO 04-07 19:57:26 [core.py:283] init engine (profile, create kv cache, warmup model) took 7.82 seconds
(EngineCore pid=7672) INFO 04-07 19:57:28 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 04-07 19:57:28 [interface.py:525] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
Model loaded.


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [10]:
# Build prompts for first 5 entries
prompts = []
for item in data[:5]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 5 questions...


Rendering prompts:   0%|          | 0/5 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/5 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


── Response 0 (id=0) ──
This is a complex or challenging question, and it is difficult to provide a direct and correct answer. I need to think about it.
Well, so I need to find the sum of the first 325 positive even whole numbers. Hmm, let's start by recalling what the first few even whole numbers are. Positive even whole numbers start at 2, right? So the first one is 2, second is 4, third is 6, fourth is 8, and so on. S ...

── Response 1 (id=1) ──
Okay, let's try to solve this integral: the integral from negative infinity to positive infinity of (a^(3/2)) divided by (s² + a²) ds. Hmm, first, I need to check if the integral converges. Since the integrand is a^(3/2)/(s² + a²), which is even in s because s² is even, so the integral from -infty to infty is twice the integral from 0 to infty. But maybe I don't need to worry about that right now. ...

── Response 2 (id=2) ──
This is a complex or challenging question, and it is difficult to provide a direct and correct answer. I need to th

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [11]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring:   0%|          | 5/1126 [00:00<00:39, 28.60it/s]

Scoring complete. 5 results.


## 8. Summary

Print accuracy broken down by question type.

In [12]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :    1 /    2  (50.00%)
  Free-form  :    2 /    3  (66.67%)
  Overall    :    3 /    5  (60.00%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [13]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 5 records to results/starter_results.jsonl
